# ARXIV CRAWLER & BENCHMARK (TỐI ƯU DUNG LƯỢNG KAGGLE)

Notebook kết hợp pipeline tải source từ `src/arxiv_crawler.ipynb` và bộ đo hiệu năng/thống kê từ `notebooks/colab_m1_selftest.ipynb` (`src.benchmark`, `src.make_stats`, `benchmark.claim.json`, `run.json`).

## Điểm chính
1. **Input & cách crawl giữ nguyên từ `src/arxiv_crawler.ipynb`**:
   - Đọc `IDS_TXT_DIR` (`arxiv_id.txt`, mỗi dòng một ID có version như `2409.18594v1`), cắt theo `[START_IDX, END_IDX)`.
   - Dùng `arxiv.Client(delay_seconds=3.0)` + `https://export.arxiv.org/src/{arxiv_id}` + `time.sleep(2)`.
   - Mỗi lượt thử 1 lần mỗi ID; chỉ retry HTTP 429 và 5xx ở lượt tiếp theo (không giới hạn số vòng).
2. **Quản lý dung lượng cho Kaggle (giới hạn 20 GiB)**:
   - File `.tar.gz` tải về và thư mục giải nén tạm được đặt trong `TEMP_DIR` (`/tmp/arxiv-crawler-temp`, nằm ngoài quota 20 GiB của `/kaggle/working`).
   - Giải nén, kiểm tra và đo dung lượng (`bytes_downloaded`, `bytes_before`, `bytes_after`) ngay sau khi tải từng bài, sau đó **xoá ngay** (`CLEANUP_MODE = "immediate"`) hoặc **xoá theo batch** (`CLEANUP_MODE = "batch"`, mỗi `BATCH_SIZE` bài) để không bao giờ tràn ổ đĩa.
3. **Output chỉ gồm các file JSON tổng hợp trong `OUTPUT_ROOT`** (không sinh hàng nghìn thư mục con `<publication-key>/v<n>/`):
   - `metadata.json`: lịch sử mọi lượt tải (`id`, `path`, `state`).
   - `status.json`: trạng thái và dung lượng chi tiết của từng Version (`version`, `kind`, `status`, `files`, `bytes_downloaded`, `bytes_before`, `bytes_after`, `seconds`).
   - `done.json`: tổng hợp theo từng Publication đã hoàn tất đủ mọi Version trong `IDS_TXT_DIR`.
   - `skipped_ids.json`: danh sách các Version không có TeX hoặc lỗi giải nén.
   - `summary.json`: các thống kê theo chuẩn `src.make_stats` (tỉ lệ thành công, tỉ lệ có TeX, độ giảm dung lượng trước/sau lọc ảnh, ...).
   - `benchmark.json`: thông số đo của `src.benchmark` (`wall_seconds`, `peak_rss_mb`, `avg_rss_mb`, `peak_disk_mb`, `final_disk_mb`, `http`, ...).
   - `benchmark.claim.json`: báo cáo chuẩn `schema_version = 1` để đối chiếu/nộp benchmark.
   - `run.json`: thông tin phiên chạy, cấu hình máy (`MACHINE`), commit/phiên bản Python, kết quả các bước giống `colab_m1_selftest.ipynb`.


In [ ]:
import tempfile
from pathlib import Path

# Kaggle: thay bằng đường dẫn file ID trong mục Input của notebook.
IDS_TXT_DIR = Path("/kaggle/input/TEN_DATASET/arxiv_id.txt")
OUTPUT_ROOT = Path("/kaggle/working/arxiv-crawler")

# Chạy local từ root repo: dùng hai dòng dưới thay cho hai dòng trên.
# IDS_TXT_DIR = Path("src/data/arxiv_id.txt")
# OUTPUT_ROOT = Path("src/data/crawler-output")

# Thư mục tạm nằm ở /tmp trên Kaggle/Linux để tránh chiếm quota 20 GiB của /kaggle/working.
TEMP_DIR = Path(tempfile.gettempdir()) / "arxiv-crawler-temp"

# Chế độ dọn file tạm sau khi đo và kiểm tra xong:
# - "immediate": xoá ngay archive + thư mục giải nén tạm sau mỗi Version (tối ưu dung lượng nhất)
# - "batch": xoá theo từng đợt BATCH_SIZE Version
CLEANUP_MODE = "immediate"  # "immediate" | "batch"
BATCH_SIZE = 50

# Lát cắt dòng ID [START_IDX, END_IDX) trong IDS_TXT_DIR
START_IDX = 0
END_IDX = 10
RETRY_ROUND_DELAY = 1

# Tham số đo hiệu năng theo chuẩn colab_m1_selftest.ipynb / src.benchmark
MEMBER = "minh"
COLD_RUN = True
BENCHMARK_INTERVAL = 1.0       # giây giữa 2 lần lấy mẫu RAM (cả cây tiến trình)
DISK_INTERVAL = 5.0            # giây giữa 2 lần đo dung lượng ổ đĩa

# Đường dẫn các file JSON đầu ra (tất cả tập trung tại OUTPUT_ROOT)
METADATA_OUTPUT_DIR = OUTPUT_ROOT / "metadata.json"
STATUS_OUTPUT_PATH = OUTPUT_ROOT / "status.json"
DONE_OUTPUT_PATH = OUTPUT_ROOT / "done.json"
SKIPPED_OUTPUT_PATH = OUTPUT_ROOT / "skipped_ids.json"
SUMMARY_OUTPUT_PATH = OUTPUT_ROOT / "summary.json"
BENCHMARK_OUTPUT_PATH = OUTPUT_ROOT / "benchmark.json"
CLAIM_OUTPUT_PATH = OUTPUT_ROOT / "benchmark.claim.json"
RUN_OUTPUT_PATH = OUTPUT_ROOT / "run.json"


In [ ]:
import arxiv
from collections import Counter
from datetime import datetime, timezone
import gzip
import json
import os
import platform
import re
import shutil
import statistics
import sys
import tarfile
import threading
import time
import urllib.request
import zlib
from pathlib import Path

import psutil
import requests

MB = 1024 * 1024
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists())
SESSION = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H%M%SZ")


def get_machine_info(work_root=OUTPUT_ROOT, temp_root=TEMP_DIR):
    """Thu thập cấu hình máy giống colab_m1_selftest.ipynb."""
    Path(work_root).mkdir(parents=True, exist_ok=True)
    Path(temp_root).mkdir(parents=True, exist_ok=True)
    return {
        "platform": platform.platform(),
        "python": sys.version,
        "cpus": os.cpu_count(),
        "ram_gib": round(psutil.virtual_memory().total / 2**30, 1),
        "output_disk_free_gib": round(shutil.disk_usage(work_root).free / 2**30, 2),
        "temp_disk_free_gib": round(shutil.disk_usage(temp_root).free / 2**30, 2),
        "kaggle": IN_KAGGLE,
    }


def dir_size(path):
    """Tổng dung lượng byte của một thư mục (qua os.walk giống src.benchmark)."""
    total = 0
    path = Path(path)
    if not path.exists():
        return 0
    for root, _dirs, files in os.walk(path):
        for name in files:
            try:
                total += os.path.getsize(os.path.join(root, name))
            except OSError:
                pass
    return total


def tree_rss(proc=None):
    """RAM (byte) của tiến trình hiện tại và toàn bộ tiến trình con (giống src.benchmark)."""
    try:
        proc = proc or psutil.Process(os.getpid())
        total = proc.memory_info().rss
        for child in proc.children(recursive=True):
            try:
                total += child.memory_info().rss
            except psutil.Error:
                pass
        return total
    except psutil.Error:
        return 0


class BenchmarkMonitor:
    """Bộ đo RAM (peak/avg RSS cả cây tiến trình) và Disk (peak/final) chạy nền."""

    def __init__(self, output_dir=OUTPUT_ROOT, temp_dir=TEMP_DIR,
                 interval=BENCHMARK_INTERVAL, disk_interval=DISK_INTERVAL):
        self.output_dir = Path(output_dir)
        self.temp_dir = Path(temp_dir)
        self.interval = interval
        self.disk_interval = disk_interval
        self.rss_values = []
        self.peak_output_disk_mb = 0.0
        self.peak_temp_disk_mb = 0.0
        self.peak_total_disk_mb = 0.0
        self.started_at = None
        self.ended_at = None
        self.started_iso = None
        self.finished_iso = None
        self.wall_seconds = 0.0
        self._t0 = None
        self._stop_event = threading.Event()
        self._thread = None
        self._proc = psutil.Process(os.getpid())

    def sample_once(self, force_disk=False):
        rss_mb = round(tree_rss(self._proc) / MB, 1)
        self.rss_values.append(rss_mb)
        if force_disk:
            out_mb = round(dir_size(self.output_dir) / MB, 2)
            tmp_mb = round(dir_size(self.temp_dir) / MB, 2)
            self.peak_output_disk_mb = max(self.peak_output_disk_mb, out_mb)
            self.peak_temp_disk_mb = max(self.peak_temp_disk_mb, tmp_mb)
            self.peak_total_disk_mb = max(self.peak_total_disk_mb, round(out_mb + tmp_mb, 2))

    def _loop(self):
        last_disk_t = -1e9
        while not self._stop_event.is_set():
            now = time.monotonic() - self._t0
            measure_disk = (now - last_disk_t) >= self.disk_interval
            if measure_disk:
                last_disk_t = now
            self.sample_once(force_disk=measure_disk)
            self._stop_event.wait(self.interval)

    def start(self):
        self.started_at = round(time.time(), 1)
        self.started_iso = datetime.now(timezone.utc).isoformat()
        self._t0 = time.monotonic()
        self._stop_event.clear()
        self.sample_once(force_disk=True)
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()

    def stop(self):
        if self._t0 is None:
            return
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=5.0)
        self.sample_once(force_disk=True)
        self.wall_seconds = round(time.monotonic() - self._t0, 2)
        self.ended_at = round(time.time(), 1)
        self.finished_iso = datetime.now(timezone.utc).isoformat()

    def summary(self, done_at_start=0, done_at_end=0):
        final_output_mb = round(dir_size(self.output_dir) / MB, 2)
        final_temp_mb = round(dir_size(self.temp_dir) / MB, 2)
        peak_output_mb = round(max(self.peak_output_disk_mb, final_output_mb), 2)
        peak_total_mb = round(max(self.peak_total_disk_mb, final_output_mb + final_temp_mb), 2)
        return {
            "wall_seconds": self.wall_seconds,
            "started_at": self.started_at,
            "ended_at": self.ended_at,
            "started_iso": self.started_iso,
            "finished_iso": self.finished_iso,
            "peak_rss_mb": max(self.rss_values) if self.rss_values else None,
            "avg_rss_mb": round(sum(self.rss_values) / len(self.rss_values), 1) if self.rss_values else None,
            "peak_disk_mb": peak_total_mb,
            "peak_output_disk_mb": peak_output_mb,
            "peak_temp_disk_mb": round(self.peak_temp_disk_mb, 2),
            "final_disk_mb": final_output_mb,
            "final_temp_disk_mb": final_temp_mb,
            "n_samples": len(self.rss_values),
            "done_papers_at_start": done_at_start,
            "done_papers_at_end": done_at_end,
        }


In [ ]:
client = arxiv.Client(delay_seconds=3.0)


def get_source(arxiv_id, save_dir=TEMP_DIR):
    """Tải archive của một Version từ export.arxiv.org/src/{arxiv_id} (giữ nguyên từ arxiv_crawler.ipynb)."""
    destination = Path(save_dir) / f"{arxiv_id}.tar.gz"
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        paper = next(client.results(arxiv.Search(id_list=[arxiv_id])))
        print(paper.title, paper.get_short_id())
        url = f"https://export.arxiv.org/src/{arxiv_id}"
        with urllib.request.urlopen(url, timeout=60) as response:
            with destination.open("wb") as output:
                shutil.copyfileobj(response, output)
        return 200
    finally:
        time.sleep(2)


def version_output(arxiv_id, save_dir=TEMP_DIR):
    """Tách ID có version thành (base_id, version_int, temp_version_dir)."""
    base_id, version = arxiv_id.rsplit("v", 1)
    publication_key = base_id.replace("/", "-").replace(".", "-")
    return base_id, int(version), Path(save_dir) / publication_key / f"v{int(version)}"


def write_json(path, data):
    """Ghi JSON nguyên tử qua file .tmp để tránh hỏng dữ liệu khi ngắt giữa chừng."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary_path.replace(path)


def read_json_if_exists(path, default):
    path = Path(path)
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return default
    return default


def dump_to_json(data: list[dict], path=METADATA_OUTPUT_DIR):
    """Nối thêm lịch sử tải vào metadata.json (giữ nguyên hợp đồng của arxiv_crawler.ipynb)."""
    if not data:
        return
    path = Path(path)
    metadata = read_json_if_exists(path, [])
    metadata.extend(data)
    write_json(path, metadata)


def inspect_and_extract_archive(arxiv_id, archive_path, temp_extract_dir, download_seconds=0.0):
    """Đọc luồng archive trực tiếp trong bộ nhớ để đo bytes_downloaded, bytes_before, bytes_after
    và lọc .tex/.bib mà không cần ghi hàng nghìn file giải nén ra ổ đĩa."""
    base_id, version, _ = version_output(arxiv_id)
    started = time.perf_counter()
    archive_size = archive_path.stat().st_size if archive_path.exists() else 0
    record = {
        "id": arxiv_id,
        "base_id": base_id,
        "version": version,
        "kind": "other",
        "status": "empty",
        "files": [],
        "bytes_downloaded": archive_size,
        "bytes_before": archive_size,
        "bytes_after": 0,
    }
    skipped = False
    try:
        with archive_path.open("rb") as source:
            signature = source.read(5)
        if signature.startswith(b"%PDF-"):
            record.update(kind="pdf", bytes_before=archive_size)
            print(f"Bỏ qua {archive_path.name}: chỉ có PDF, không có nguồn LaTeX")
            skipped = True
            return record, skipped

        if tarfile.is_tarfile(archive_path):
            record["kind"] = "tar"
            kept_files = {}
            with tarfile.open(archive_path, "r:*") as archive:
                all_members = archive.getmembers()
                record["bytes_before"] = sum(m.size for m in all_members if m.isfile())
                for member in all_members:
                    if not member.isfile():
                        continue
                    if Path(member.name).suffix.lower() not in {".tex", ".bib"}:
                        continue
                    # Kiểm tra an toàn đường dẫn giống archive.extractall(..., filter="data")
                    filtered = tarfile.data_filter(member, str(temp_extract_dir))
                    if filtered is None:
                        continue
                    extracted = archive.extractfile(filtered)
                    member_bytes = 0
                    if extracted is not None:
                        with extracted:
                            for chunk in iter(lambda: extracted.read(1024 * 1024), b""):
                                member_bytes += len(chunk)
                    rel_name = str(Path(member.name.lstrip("/")))
                    kept_files[rel_name] = member_bytes
            record["files"] = list(kept_files.keys())
            record["bytes_after"] = sum(kept_files.values())
            kept_count = len(record["files"])
        elif signature.startswith(b"\x1f\x8b"):
            with gzip.open(archive_path, "rb") as source:
                prefix = source.read(16384)
                total_uncompressed = len(prefix)
                for chunk in iter(lambda: source.read(1024 * 1024), b""):
                    total_uncompressed += len(chunk)
                if prefix.startswith(b"%PDF-"):
                    record.update(kind="pdf", bytes_before=total_uncompressed)
                    print(f"Bỏ qua {archive_path.name}: GZIP chứa PDF")
                    skipped = True
                    return record, skipped
                if not any(marker in prefix for marker in (
                    b"\\documentclass", b"\\documentstyle", b"\\begin{document}", b"\\input"
                )):
                    record["bytes_before"] = total_uncompressed
                    print(f"Bỏ qua {archive_path.name}: GZIP không nhận diện được nguồn LaTeX")
                    skipped = True
                    return record, skipped
            kept_count = 1
            record.update(
                kind="gz_single",
                files=["main.tex"],
                bytes_before=total_uncompressed,
                bytes_after=total_uncompressed,
            )
        else:
            print(f"Bỏ qua {archive_path.name}: định dạng không hỗ trợ")
            skipped = True
            return record, skipped

        record["status"] = "ok" if record["files"] else "empty"
        print(f"Đã xử lý {archive_path.name}: giữ {kept_count} file .tex/.bib ({record['bytes_after']} bytes)")
    except (tarfile.TarError, OSError, EOFError, zlib.error) as error:
        record.update(status="error", error=type(error).__name__, files=[], bytes_after=0)
        skipped = True
        print(f"Không giải nén được {archive_path.name}: {error}")
    finally:
        record["seconds"] = round(download_seconds + (time.perf_counter() - started), 3)
    return record, skipped


def cleanup_temp_items(items):
    """Xoá danh sách (archive_path, temp_extract_dir) trong thư mục tạm để giải phóng dung lượng."""
    for archive_path, extract_dir in items:
        if archive_path is not None:
            Path(archive_path).unlink(missing_ok=True)
        if extract_dir is not None and Path(extract_dir).exists():
            shutil.rmtree(extract_dir, ignore_errors=True)
            parent = Path(extract_dir).parent
            if parent.exists() and not any(parent.iterdir()):
                parent.rmdir()
    items.clear()


In [ ]:
def load_expected_publications(ids_txt_path=IDS_TXT_DIR):
    """Đọc toàn bộ file ID để biết mỗi Publication có bao nhiêu Version dự kiến."""
    publications = {}
    path = Path(ids_txt_path)
    if not path.exists():
        return publications
    for line in path.read_text(encoding="utf-8").splitlines():
        version_id = line.strip()
        if version_id:
            base_id, version, _ = version_output(version_id)
            publications.setdefault(base_id, []).append((version, version_id))
    for base_id in publications:
        publications[base_id].sort()
    return publications


def compute_done_map(status_map, ids_txt_path=IDS_TXT_DIR):
    """Tổng hợp done.json cho các Publication đã xử lý xong đủ mọi Version trong file ID."""
    publications = load_expected_publications(ids_txt_path)
    done_map = {}
    for base_id, version_items in publications.items():
        records = []
        for _version_num, version_id in version_items:
            record = status_map.get(version_id)
            if not record or record.get("status") not in {"ok", "empty"}:
                break
            records.append(record)
        if len(records) != len(version_items):
            continue
        done_map[base_id] = {
            "id": base_id,
            "status": "done",
            "n_versions": len(records),
            "versions_with_tex": sum(r["status"] == "ok" for r in records),
            "versions_non_tex": sum(r["status"] == "empty" for r in records),
            "bytes_downloaded": sum(r.get("bytes_downloaded", 0) for r in records),
            "bytes_before": sum(r.get("bytes_before", 0) for r in records),
            "bytes_after": sum(r.get("bytes_after", 0) for r in records),
            "seconds": round(sum(r.get("seconds", 0.0) for r in records), 3),
        }
    return done_map


def segments_for(base_ids):
    """Gộp các base_id dạng YYMM.NNNNN thành range.segments giống colab_m1_selftest.ipynb."""
    parsed = []
    for bid in base_ids:
        if "." in bid:
            prefix, num_str = bid.split(".", 1)
            if prefix.isdigit() and num_str.isdigit():
                parsed.append((prefix, int(num_str)))
    segments = []
    for prefix, number in sorted(set(parsed)):
        if segments and segments[-1]["prefix"] == prefix and segments[-1]["end"] == number - 1:
            segments[-1]["end"] = number
        else:
            segments.append({"prefix": prefix, "start": number, "end": number})
    return segments


def _avg(values):
    vals = [v for v in values if v is not None]
    return round(statistics.mean(vals), 3) if vals else None


def _ratio(part, whole):
    return round(part / whole, 4) if whole else None


def build_summary_stats(selected_ids, status_map, done_map, ids_txt_path=IDS_TXT_DIR):
    """Tính toán bộ chỉ số summary.json tương đương src.make_stats trên các ID đã chạy."""
    all_publications = load_expected_publications(ids_txt_path)
    selected_base_ids = list(dict.fromkeys(version_output(vid)[0] for vid in selected_ids))

    papers_done = []
    papers_error = []
    papers_incomplete = []
    papers_not_attempted = []

    for base_id in selected_base_ids:
        expected_versions = [vid for _v, vid in all_publications.get(base_id, [])]
        if not expected_versions:
            expected_versions = [vid for vid in selected_ids if version_output(vid)[0] == base_id]
        recs = [status_map.get(vid) for vid in expected_versions]
        if all(r is None for r in recs):
            papers_not_attempted.append(base_id)
        elif base_id in done_map:
            papers_done.append(done_map[base_id])
        elif any(r is not None and r.get("status") == "error" for r in recs):
            papers_error.append(base_id)
        else:
            papers_incomplete.append(base_id)

    processed_versions = [status_map[vid] for vid in selected_ids if vid in status_map]
    versions_total = len(processed_versions)
    versions_with_tex = sum(r.get("status") == "ok" for r in processed_versions)
    versions_non_tex = sum(r.get("status") == "empty" for r in processed_versions)
    versions_failed = sum(r.get("status") == "error" for r in processed_versions)

    total_bytes_downloaded = sum(r.get("bytes_downloaded", 0) for r in processed_versions)
    total_bytes_before = sum(r.get("bytes_before", 0) for r in processed_versions if r.get("status") in {"ok", "empty"})
    total_bytes_after = sum(r.get("bytes_after", 0) for r in processed_versions if r.get("status") in {"ok", "empty"})

    sized_papers = [p for p in papers_done if p.get("bytes_before") is not None]
    before_sum_done = sum(p["bytes_before"] for p in sized_papers)
    after_sum_done = sum(p["bytes_after"] for p in sized_papers)

    # Nếu lát cắt có các bài hoàn tất thì tính reducción trên tập done giống make_stats;
    # đồng thời cung cấp cả tổng byte trên các Version đã xử lý.
    reduction_base = before_sum_done if before_sum_done > 0 else total_bytes_before
    reduction_after = after_sum_done if before_sum_done > 0 else total_bytes_after

    return {
        "papers_requested": len(selected_base_ids),
        "papers_done": len(papers_done),
        "papers_error": len(papers_error),
        "papers_incomplete": len(papers_incomplete),
        "papers_not_attempted": len(papers_not_attempted),
        "paper_success_rate": _ratio(len(papers_done), len(selected_base_ids)),
        "versions_requested": len(selected_ids),
        "versions_total": versions_total,
        "versions_with_tex": versions_with_tex,
        "versions_non_tex": versions_non_tex,
        "versions_failed": versions_failed,
        "version_success_rate": _ratio(versions_with_tex + versions_non_tex, len(selected_ids)),
        "version_tex_rate": _ratio(versions_with_tex, versions_total),
        "total_bytes_downloaded": total_bytes_downloaded,
        "total_bytes_before": total_bytes_before,
        "total_bytes_after": total_bytes_after,
        "avg_bytes_downloaded_per_paper": _avg([p["bytes_downloaded"] for p in sized_papers]),
        "avg_bytes_before_per_paper": _avg([p["bytes_before"] for p in sized_papers]),
        "avg_bytes_after_per_paper": _avg([p["bytes_after"] for p in papers_done]),
        "papers_with_size_before": len(sized_papers),
        "size_reduction_pct": (
            round(100.0 * (1.0 - reduction_after / reduction_base), 2)
            if reduction_base else None
        ),
    }


In [ ]:
def get_id_list():
    """Đọc lát cắt [START_IDX, END_IDX) từ file IDS_TXT_DIR (giữ nguyên từ arxiv_crawler.ipynb)."""
    id_list = []
    with Path(IDS_TXT_DIR).open("r", encoding="utf-8") as file:
        for index, line in enumerate(file):
            if index >= END_IDX:
                break
            if index >= START_IDX and line.strip():
                id_list.append(line.strip())
    return id_list


id_list = get_id_list()
print(f"Đã nạp {len(id_list)} Version ID từ [{START_IDX}, {END_IDX})")

In [ ]:
def save_checkpoint_jsons(status_map, skipped_ids, ids_txt_path=IDS_TXT_DIR):
    """Lưu đồng bộ status.json, done.json và skipped_ids.json."""
    done_map = compute_done_map(status_map, ids_txt_path=ids_txt_path)
    write_json(STATUS_OUTPUT_PATH, status_map)
    write_json(DONE_OUTPUT_PATH, done_map)
    write_json(SKIPPED_OUTPUT_PATH, list(dict.fromkeys(skipped_ids)))
    return done_map


def crawl_sources():
    """Tải source, giải nén + đo trong TEMP_DIR, xoá tạm ngay/theo batch và xuất tất cả JSON."""
    Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
    Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

    machine = get_machine_info(OUTPUT_ROOT, TEMP_DIR)
    status_map = read_json_if_exists(STATUS_OUTPUT_PATH, {})
    skipped_ids = read_json_if_exists(SKIPPED_OUTPUT_PATH, [])
    initial_done = compute_done_map(status_map, ids_txt_path=IDS_TXT_DIR)

    monitor = BenchmarkMonitor(
        output_dir=OUTPUT_ROOT,
        temp_dir=TEMP_DIR,
        interval=BENCHMARK_INTERVAL,
        disk_interval=DISK_INTERVAL,
    )
    monitor.start()

    count = 0
    requests_count = 0
    retries_count = 0
    http_429_count = 0
    http_5xx_count = 0
    backoff_seconds = 0.0
    errors_by_type = Counter()

    metadata_list = []
    temp_cleanup_queue = []
    pending_ids = list(id_list)
    retry_round = 0

    try:
        while pending_ids:
            retry_ids = []
            for arxiv_id in pending_ids:
                count += 1
                requests_count += 1
                if retry_round > 0:
                    retries_count += 1

                base_id, version, temp_version_dir = version_output(arxiv_id, save_dir=TEMP_DIR)
                archive_path = Path(TEMP_DIR) / f"{arxiv_id}.tar.gz"
                started = time.perf_counter()

                record = {
                    "id": arxiv_id,
                    "base_id": base_id,
                    "version": version,
                    "kind": "download_error",
                    "status": "error",
                    "files": [],
                    "bytes_downloaded": 0,
                    "bytes_before": 0,
                    "bytes_after": 0,
                }
                state = "fail"
                try:
                    res = get_source(arxiv_id=arxiv_id, save_dir=TEMP_DIR)
                    state = "success" if res == 200 else "fail"
                except (OSError, StopIteration, arxiv.ArxivError, requests.exceptions.RequestException) as error:
                    err_name = type(error).__name__
                    print(f"Không tải được {arxiv_id}: {err_name}")
                    state = "fail"
                    record["error"] = err_name
                    http_status = getattr(error, "code", getattr(error, "status", None))
                    if isinstance(error, requests.exceptions.HTTPError) and error.response is not None:
                        http_status = error.response.status_code
                    if http_status is not None:
                        errors_by_type[str(http_status)] += 1
                    else:
                        errors_by_type[err_name] += 1

                    if http_status == 429 or (http_status is not None and 500 <= http_status < 600):
                        if http_status == 429:
                            http_429_count += 1
                        else:
                            http_5xx_count += 1
                        retry_ids.append(arxiv_id)
                        print(f"[RETRY-QUEUED] {arxiv_id}: thử lại sau khi kết thúc lượt hiện tại")

                download_elapsed = round(time.perf_counter() - started, 3)
                record["seconds"] = download_elapsed

                # Nếu tải thành công, kiểm tra/giải nén ngay trong TEMP_DIR để lấy số đo dung lượng
                if state == "success" and archive_path.exists():
                    record, was_skipped = inspect_and_extract_archive(
                        arxiv_id=arxiv_id,
                        archive_path=archive_path,
                        temp_extract_dir=temp_version_dir,
                        download_seconds=download_elapsed,
                    )
                    if record.get("status") == "error" and record.get("error"):
                        errors_by_type[record["error"]] += 1
                    if was_skipped and arxiv_id not in skipped_ids:
                        skipped_ids.append(arxiv_id)
                    elif not was_skipped and arxiv_id in skipped_ids:
                        skipped_ids.remove(arxiv_id)

                # Cập nhật mẫu dung lượng đĩa đỉnh trước khi xoá file tạm
                monitor.sample_once(force_disk=True)

                temp_cleanup_queue.append((archive_path, temp_version_dir))
                if CLEANUP_MODE == "immediate":
                    cleanup_temp_items(temp_cleanup_queue)
                elif CLEANUP_MODE == "batch" and len(temp_cleanup_queue) >= BATCH_SIZE:
                    cleanup_temp_items(temp_cleanup_queue)

                status_map[arxiv_id] = record
                metadata_list.append({"id": arxiv_id, "path": str(archive_path), "state": state})

                if count % BATCH_SIZE == 0:
                    dump_to_json(metadata_list)
                    metadata_list.clear()
                    save_checkpoint_jsons(status_map, skipped_ids, ids_txt_path=IDS_TXT_DIR)

            dump_to_json(metadata_list)
            metadata_list.clear()
            cleanup_temp_items(temp_cleanup_queue)
            save_checkpoint_jsons(status_map, skipped_ids, ids_txt_path=IDS_TXT_DIR)

            pending_ids = retry_ids
            if pending_ids:
                retry_round += 1
                print(f"[RETRY] round={retry_round}; pending={len(pending_ids)}; wait={RETRY_ROUND_DELAY}s")
                time.sleep(RETRY_ROUND_DELAY)
                backoff_seconds = round(backoff_seconds + RETRY_ROUND_DELAY, 3)
    finally:
        dump_to_json(metadata_list)
        metadata_list.clear()
        cleanup_temp_items(temp_cleanup_queue)
        if Path(TEMP_DIR).exists():
            shutil.rmtree(TEMP_DIR, ignore_errors=True)

        done_map = save_checkpoint_jsons(status_map, skipped_ids, ids_txt_path=IDS_TXT_DIR)
        summary = build_summary_stats(id_list, status_map, done_map, ids_txt_path=IDS_TXT_DIR)
        write_json(SUMMARY_OUTPUT_PATH, summary)

        monitor.stop()
        bench = monitor.summary(done_at_start=len(initial_done), done_at_end=len(done_map))
        http_stats = {
            "requests": requests_count,
            "retries": retries_count,
            "retry_rounds": retry_round,
            "http_429": http_429_count,
            "http_5xx": http_5xx_count,
            "backoff_seconds": backoff_seconds,
            "errors_by_type": dict(errors_by_type),
        }
        bench["http"] = http_stats
        write_json(BENCHMARK_OUTPUT_PATH, bench)

        base_ids = list(dict.fromkeys(version_output(vid)[0] for vid in id_list))
        claim = {
            "schema_version": 1,
            "member": MEMBER,
            "range": {"segments": segments_for(base_ids)},
            "ids_attempted": len(base_ids),
            "run": {
                "cold_run": COLD_RUN,
                "resumed_passes": 0 if COLD_RUN else 1,
                "started_at": bench["started_iso"],
                "finished_at": bench["finished_iso"],
                "machine": machine["platform"],
                "python": machine["python"],
                "s2_api_key": False,
                "measured_by": "BenchmarkMonitor (psutil; RSS cả cây tiến trình, disk qua os.walk)",
            },
            "stages": {
                "discovery": {"seconds": None, "requests": None, "retries": None, "errors_by_type": {}},
                "source": {
                    "seconds": bench["wall_seconds"],
                    "requests": requests_count,
                    "retries": retries_count,
                    "errors_by_type": dict(errors_by_type),
                },
                "references": {"seconds": None, "requests": None, "retries": None, "errors_by_type": {}},
                "export": {"seconds": None},
            },
            "totals": {
                "wall_seconds": bench["wall_seconds"],
                "publications_found": len(base_ids),
                "publications_complete": summary["papers_done"],
                "versions_expected": summary["versions_requested"],
                "versions_ok": summary["versions_with_tex"] + summary["versions_non_tex"],
                "versions_with_tex": summary["versions_with_tex"],
                "versions_non_tex": summary["versions_non_tex"],
                "versions_failed": summary["versions_failed"],
                "references_total": None,
                "references_with_arxiv_id": None,
                "publications_s2_ok": None,
                "bytes_downloaded": summary["total_bytes_downloaded"],
                "bytes_before_figure_removal": summary["total_bytes_before"],
                "bytes_after_figure_removal": summary["total_bytes_after"],
                "size_reduction_pct": summary["size_reduction_pct"],
                "peak_rss_mb": bench["peak_rss_mb"],
                "avg_rss_mb": bench["avg_rss_mb"],
                "peak_disk_mb": bench["peak_disk_mb"],
                "output_mb": bench["final_disk_mb"],
            },
            "http": http_stats,
            "notes": f"Kaggle/local crawl slice [{START_IDX}, {END_IDX}), cleanup_mode={CLEANUP_MODE}",
        }
        write_json(CLAIM_OUTPUT_PATH, claim)

        run_payload = {
            "session": SESSION,
            "member": MEMBER,
            "slice": {"start_idx": START_IDX, "end_idx": END_IDX, "versions_count": len(id_list)},
            "cleanup_mode": CLEANUP_MODE,
            "batch_size": BATCH_SIZE,
            "machine": machine,
            "benchmark": bench,
            "http": http_stats,
            "summary": summary,
            "outputs": {
                "metadata": str(METADATA_OUTPUT_DIR),
                "status": str(STATUS_OUTPUT_PATH),
                "done": str(DONE_OUTPUT_PATH),
                "skipped_ids": str(SKIPPED_OUTPUT_PATH),
                "summary": str(SUMMARY_OUTPUT_PATH),
                "benchmark": str(BENCHMARK_OUTPUT_PATH),
                "claim": str(CLAIM_OUTPUT_PATH),
            },
        }
        write_json(RUN_OUTPUT_PATH, run_payload)

    return {
        "summary": summary,
        "benchmark": bench,
        "claim": claim,
        "run": run_payload,
    }


results = crawl_sources()


In [ ]:
# Xem nhanh kết quả đo hiệu năng và thống kê sau lượt chạy
print("=== TỔNG HỢP HIỆU NĂNG (benchmark.json) ===")
print(json.dumps(results["benchmark"], ensure_ascii=False, indent=2))

print("\n=== THỐNG KÊ DỮ LIỆU (summary.json) ===")
print(json.dumps(results["summary"], ensure_ascii=False, indent=2))

print("\n=== CÁC FILE JSON ĐÃ XUẤT TRONG OUTPUT_ROOT ===")
for json_file in sorted(Path(OUTPUT_ROOT).glob("*.json")):
    print(f"- {json_file.name}: {json_file.stat().st_size:,} bytes")
